In [0]:
%run ../init_framework

In [0]:
# # 1. Initialize the CDF Read Stream  
# df_bronze_loans = (spark.readStream
#     .format("delta")
#     .option("readChangeFeed", "true")
#     .option("startingVersion", 1) # Required for the very first execution
#     .table(LOANS_BRONZE))


In [0]:
# 1. Initialize the CDF Read Stream  
df_bronze_loans = (spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) # Required for the very first execution
    .table(LOANS_BRONZE))
df_bronze_loans.limit(3).display()

In [0]:
LOAN_RENAME_MAP = {
    "loan_amnt": "loan_amount",
    "funded_amnt": "funded_amount",
    "term": "loan_term_months",
    "int_rate": "interest_rate",
    "installment": "monthly_installment",
    "issue_d": "issue_date",
    "loan_status": "loan_status",
    "purpose": "loan_purpose"
}

# --- Execution in Notebook ---
df_renamed_loans = standardize_column_names(df_bronze_loans, LOAN_RENAME_MAP, strict=True)

In [0]:
df_with_metadata = apply_silver_metadata(df_renamed_loans)

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
df_distinct.count()

In [0]:
# Keep Valid Loans: Drop NULL or UNKNOWN Loans
col_ls = ["loan_amount", "funded_amount", "loan_term_months", "interest_rate", "monthly_installment", "issue_date", "loan_status", "loan_purpose"]
df_valid_loans = drop_critical_nulls(df_distinct, col_ls)

In [0]:
df_valid_loans.count()

In [0]:
# Convert loan_term_months to loan_term_years and cast as INT

from pyspark.sql.functions import col, regexp_replace

df_loan_years = df_valid_loans.withColumn(
    "loan_term_years", 
    (regexp_replace(col("loan_term_months"), r"\D+", "").cast("int") / 12).cast("int")
).drop("loan_term_months")

In [0]:
from pyspark.sql.functions import col, coalesce, lit, broadcast

# Read the lookup table for valid codes
df_lkp = spark.read.table(REF_LOAN_PURPOSES).filter("is_active = true")

# Join with aliases to manage the "loan_purpose" column from both sides
df_joined = df_loan_years.alias("base").join(
    broadcast(df_lkp.alias("lkp")), 
    col("base.loan_purpose") == col("lkp.loan_purpose"), 
    "left"
).drop(col("base.loan_purpose"))

# 3. Simple Logic: If is_active is NULL, the purpose wasn't in our active list
df_loans_final = (df_joined
    .withColumn("loan_purpose", coalesce(col("lkp.loan_purpose"), lit("other")))
    .drop("is_active")
)

In [0]:
df_loans_final.limit(3).display()